# 12.12 答案需要同時看圖和聽聲音時怎麼辦？


給一張圓形圖和一段高音，要求回答「circle,high」。圖片可以告訴形狀，卻不能告訴音高；聲音告訴音高，卻沒有圖片形狀。這樣的任務讓兩種輸入各提供一項必需線索，避免模型靠單一模態完成全部答案。本節先檢查兩條路都接到文字代價。

前置是[10.7圖片展開](https://birdhackor.github.io/tiny-perceptron-vlm/10.7.html)、[12.9聲音展開](https://birdhackor.github.io/tiny-perceptron-vlm/12.9.html)與[11.8按規則替換](https://birdhackor.github.io/tiny-perceptron-vlm/11.8.html)。模態modality指資訊來源的種類，這裡是圖與聲音。答案由兩個部分組成，聯合exact match要求兩部分同時正確；單項正確率則分別看形狀與音高。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.data import ByteTokenizer
from tiny_perceptron.model import TinyLM, ModelConfig, masked_loss
from tiny_perceptron.multimodal import MultiModalLM, scene, tone

torch.manual_seed(0)
tok = ByteTokenizer()
model = MultiModalLM(TinyLM(ModelConfig(width=8)))
prefix = [tok.bos_id, tok.user_id, tok.image_id, tok.audio_id, tok.eos_id, tok.assistant_id]
answer = tok.encode("circle,high") + [tok.eos_id]
ids = torch.tensor(prefix + answer)
labels = torch.tensor([-100] * len(prefix) + answer)
out = model(ids, labels, image=scene("red", "circle"), waveform=tone(440))
masked_loss(out["logits"], out["labels"]).backward()
print("圖路徑梯度", model.image_projector.weight.grad.norm().item() > 0)
print("音路徑梯度", model.audio_projector.weight.grad.norm().item() > 0)

輸入同時有兩種標記與對應資料，人工目標circle,high遵守300Hz高低邊界。模型將圖片標記換16條特徵、聲音標記換11條特徵，再對完整展開序列做一次錯位。兩個接頭都在回答預測的計算鏈中，兩行通常True；不過梯度來自整個字串代價，不表示它已學會把形狀只歸圖、音高只歸聲。

這段沒有更新參數，真正學習應包含circle,low、circle,high、square,low、square,high等組合，留新組合或新頻率位置用於測試。若圓形總搭高音，模型可能從圖片猜兩項，違背原本設計的聯合需求。

訓練後做兩種替換：保持圖與問句、只換高低音，理想只改逗號後那項；保持聲音與問句、只換圓方圖，理想只改前項。再分別報告兩項與整串成功率。

聯合任務的猜測基準比單項更難：形狀圓方各半、音高高低各半且獨立時，只猜circle,high約有四分之一整串正確；只猜對形狀但音高隨機，也不能叫成功融合。這裡獨立與平衡要由資料設計保證，如果所有圓圖都配高音，一個模態就可能推兩項。測量時保存每筆的形狀目標、音高目標和完整輸出，讓讀者知道錯誤集中哪一項。梯度兩條都非零只是證明計算能經過它們，真正語義分工要靠後續保持一項、替換另一項的測試來確認。

練習先將錄音換200Hz，目標只改成circle,low；說明哪些輸入與答案部分保持不變，再執行核對兩條梯度路徑仍存在。要證明正確利用兩種資訊，仍需之後的實際回答測試。
